# Plaka Tespiti Eğitimi — YOLO11m (Colab)

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**.

Veri seti: `keremberke/license-plate-object-detection` (HuggingFace, ~8.8k görsel, tek sınıf: plate).
Eğitim çıktıları Google Drive'a yazılır; Colab bağlantısı koparsa **Devam et** hücresiyle kaldığı yerden sürer.

In [ ]:
!nvidia-smi
!pip -q install ultralytics huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/arabica_hackathon/runs_plate'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

## 1) Veri setini indir ve YOLO formatına çevir

In [ ]:
import argparse
import json
import shutil
import zipfile
from pathlib import Path

ROOT = Path("/content")
DS_DIR = ROOT / "datasets" / "plates"
REPO = "keremberke/license-plate-object-detection"
SPLITS = {"train": "train", "valid": "val", "test": "test"}


def download_and_extract():
    from huggingface_hub import hf_hub_download

    raw = DS_DIR / "_raw"
    raw.mkdir(parents=True, exist_ok=True)
    for split in SPLITS:
        out = raw / split
        if out.exists() and any(out.rglob("*")):
            print(f"[veri] {split} zaten var, atlanıyor")
            continue
        print(f"[veri] {split}.zip indiriliyor...")
        zp = hf_hub_download(REPO, f"data/{split}.zip", repo_type="dataset")
        with zipfile.ZipFile(zp) as z:
            z.extractall(out)
    return raw


def coco_to_yolo(raw: Path):
    """Her split'teki COCO json'u YOLO formatına çevirir. Tek sınıf: plate."""
    for split, ysplit in SPLITS.items():
        img_out = DS_DIR / "images" / ysplit
        lbl_out = DS_DIR / "labels" / ysplit
        if img_out.exists() and any(img_out.iterdir()):
            print(f"[dönüşüm] {ysplit} zaten hazır")
            continue
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        jsons = list((raw / split).rglob("*.json"))
        if not jsons:
            raise FileNotFoundError(f"{raw / split} içinde COCO json bulunamadı")
        ann_path = jsons[0]
        coco = json.loads(ann_path.read_text(encoding="utf-8"))
        img_root = ann_path.parent

        images = {im["id"]: im for im in coco["images"]}
        boxes = {}
        for a in coco["annotations"]:
            boxes.setdefault(a["image_id"], []).append(a["bbox"])

        n = 0
        for iid, im in images.items():
            src = img_root / im["file_name"]
            if not src.exists():
                hits = list(img_root.rglob(Path(im["file_name"]).name))
                if not hits:
                    continue
                src = hits[0]
            W, H = im["width"], im["height"]
            lines = []
            for x, y, w, h in boxes.get(iid, []):
                if w <= 1 or h <= 1:
                    continue
                cx, cy = (x + w / 2) / W, (y + h / 2) / H
                lines.append(f"0 {cx:.6f} {cy:.6f} {w / W:.6f} {h / H:.6f}")
            dst = img_out / src.name
            shutil.copy2(src, dst)
            (lbl_out / (src.stem + ".txt")).write_text("\n".join(lines))
            n += 1
        print(f"[dönüşüm] {ysplit}: {n} görsel")

    yaml = DS_DIR / "plates.yaml"
    yaml.write_text(
        f"path: {DS_DIR.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        "names:\n  0: plate\n",
        encoding="utf-8",
    )
    return yaml

raw = download_and_extract()
DATA_YAML = coco_to_yolo(raw)
print(DATA_YAML.read_text())

## 2) Eğitim
`batch=-1` GPU belleğine göre batch boyutunu otomatik seçer. T4'te epoch başına birkaç dakika sürer; kalan süreyi ilk epoch'tan sonra kontrol edin.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11m.pt')
model.train(
    data=str(DATA_YAML), epochs=40, imgsz=640, batch=-1, workers=2,
    patience=10, cos_lr=True, amp=True, device=0,
    project=SAVE_DIR, name='yolo11m_plate', exist_ok=True, plots=True,
    save_period=5,
)

## 2b) Devam et (yalnızca bağlantı koptuysa)
Önce 1. adımı (Drive bağlama + veri) tekrar çalıştırın, sonra bu hücreyi.

In [ ]:
from ultralytics import YOLO
YOLO(f'{SAVE_DIR}/yolo11m_plate/weights/last.pt').train(resume=True)

## 3) Test seti değerlendirmesi ve indirme

In [ ]:
from ultralytics import YOLO
best = f'{SAVE_DIR}/yolo11m_plate/weights/best.pt'
m = YOLO(best)
metrics = m.val(data=str(DATA_YAML), split='test')
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)
from google.colab import files
files.download(best)